# 02 - Feature Extraction
Tüm .wav dosyalarından MFCC + Delta MFCC çıkar, train/test olarak böl ve kaydet.

In [1]:
import os, glob, json
import numpy as np
import librosa
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')
print('Kütüphaneler yüklendi ✓')

Kütüphaneler yüklendi ✓


In [2]:
# ── AYARLAR ──────────────────────────────────────────
BASE_DIR     = r'C:\Users\elifs\OneDrive\Masaüstü\acoustiguard_final'
MACHINE_TYPE = 'fan'
MODEL_IDS    = ['id_00', 'id_02', 'id_04', 'id_06']
SAMPLE_RATE  = 16000
CHANNEL      = 0
N_MFCC       = 40
N_FFT        = 1024
HOP_LENGTH   = 512
TEST_SIZE    = 0.2   # normal verinin %20'si test'e gider
RANDOM_STATE = 42
# ─────────────────────────────────────────────────────

OUT_DIR = os.path.join(BASE_DIR, 'outputs', 'features', MACHINE_TYPE)
os.makedirs(OUT_DIR, exist_ok=True)
print(f'Çıktı: {OUT_DIR}')

Çıktı: C:\Users\elifs\OneDrive\Masaüstü\acoustiguard_final\outputs\features\fan


## 1. Fonksiyonlar

In [3]:
def load_audio(fp):
    y, _ = librosa.load(fp, sr=SAMPLE_RATE, mono=False)
    return y[CHANNEL] if y.ndim > 1 else y

def extract_features(y):
    """MFCC + Delta MFCC → 160 boyutlu vektör"""
    mfcc  = librosa.feature.mfcc(y=y, sr=SAMPLE_RATE, n_mfcc=N_MFCC,
                                  n_fft=N_FFT, hop_length=HOP_LENGTH)
    delta = librosa.feature.delta(mfcc)
    return np.concatenate([
        np.mean(mfcc,  axis=1), np.std(mfcc,  axis=1),
        np.mean(delta, axis=1), np.std(delta, axis=1),
    ])

print('Fonksiyonlar hazır ✓  (feature boyutu: 160)')

Fonksiyonlar hazır ✓  (feature boyutu: 160)


## 2. Tüm Model ID'ler için Feature Extraction

In [4]:
all_results = {}

for model_id in MODEL_IDS:
    print(f'\n── {model_id} ──────────────────────')
    machine_dir = os.path.join(BASE_DIR, 'data', 'raw', 'mimii', MACHINE_TYPE, model_id)

    normal_files   = sorted(glob.glob(os.path.join(machine_dir, 'normal',   '*.wav')))
    abnormal_files = sorted(glob.glob(os.path.join(machine_dir, 'abnormal', '*.wav')))

    if not normal_files:
        print(f'  [ATLA] Dosya bulunamadı')
        continue

    # Normal: %80 train, %20 test
    train_files, test_normal_files = train_test_split(
        normal_files, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    print(f'  Train normal   : {len(train_files)}')
    print(f'  Test normal    : {len(test_normal_files)}')
    print(f'  Test abnormal  : {len(abnormal_files)}')

    def process(files, label, desc):
        X, y = [], []
        for fp in tqdm(files, desc=desc, leave=False):
            try:
                X.append(extract_features(load_audio(fp)))
                y.append(label)
            except Exception as e:
                print(f'  [HATA] {fp}: {e}')
        return np.array(X), np.array(y)

    X_train, y_train = process(train_files,       0, 'train normal')
    X_tn,    y_tn    = process(test_normal_files,  0, 'test normal')
    X_ta,    y_ta    = process(abnormal_files,     1, 'test abnormal')

    X_test = np.vstack([X_tn, X_ta])
    y_test = np.concatenate([y_tn, y_ta])

    all_results[model_id] = {
        'X_train': X_train, 'y_train': y_train,
        'X_test':  X_test,  'y_test':  y_test,
        'n_test_normal': len(y_tn), 'n_test_abnormal': len(y_ta)
    }
    print(f'  X_train: {X_train.shape} | X_test: {X_test.shape}')

print('\n✓ Feature extraction tamamlandı!')


── id_00 ──────────────────────
  Train normal   : 808
  Test normal    : 203
  Test abnormal  : 407


train normal:   0%|          | 0/808 [00:00<?, ?it/s]

test normal:   0%|          | 0/203 [00:00<?, ?it/s]

test abnormal:   0%|          | 0/407 [00:00<?, ?it/s]

  X_train: (808, 160) | X_test: (610, 160)

── id_02 ──────────────────────
  Train normal   : 812
  Test normal    : 204
  Test abnormal  : 359


...

  X_train: (826, 160) | X_test: (555, 160)

── id_06 ──────────────────────
  Train normal   : 812
  Test normal    : 203
  Test abnormal  : 361


train normal:   0%|          | 0/812 [00:00<?, ?it/s]

test normal:   0%|          | 0/203 [00:00<?, ?it/s]

test abnormal:   0%|          | 0/361 [00:00<?, ?it/s]

  X_train: (812, 160) | X_test: (564, 160)

✓ Feature extraction tamamlandı!


## 3. Kaydet

In [5]:
for model_id, data in all_results.items():
    save_dir = os.path.join(OUT_DIR, model_id)
    os.makedirs(save_dir, exist_ok=True)
    np.save(os.path.join(save_dir, 'X_train.npy'), data['X_train'])
    np.save(os.path.join(save_dir, 'y_train.npy'), data['y_train'])
    np.save(os.path.join(save_dir, 'X_test.npy'),  data['X_test'])
    np.save(os.path.join(save_dir, 'y_test.npy'),  data['y_test'])
    meta = {'n_test_normal': data['n_test_normal'], 'n_test_abnormal': data['n_test_abnormal']}
    with open(os.path.join(save_dir, 'meta.json'), 'w') as f:
        json.dump(meta, f)
    print(f'{model_id} kaydedildi → {save_dir}')

print('\n➡️  Sıradaki: 03_train_evaluate.ipynb')

id_00 kaydedildi → C:\Users\elifs\OneDrive\Masaüstü\acoustiguard_final\outputs\features\fan\id_00
id_02 kaydedildi → C:\Users\elifs\OneDrive\Masaüstü\acoustiguard_final\outputs\features\fan\id_02
id_04 kaydedildi → C:\Users\elifs\OneDrive\Masaüstü\acoustiguard_final\outputs\features\fan\id_04
id_06 kaydedildi → C:\Users\elifs\OneDrive\Masaüstü\acoustiguard_final\outputs\features\fan\id_06

➡️  Sıradaki: 03_train_evaluate.ipynb
